# Lecture 1 examples: randomness, data, and probability models

This notebook accompanies the blackboard lecture. We simulate specified probability models and study their consequences; fitting models to observed data belongs to the separate statistics course. The course prioritizes complete proofs over a broad catalogue of topics. This notebook illustrates the mathematical arguments; it does not replace them. It makes three distinctions visible:

1. empirical frequencies fluctuate in finite samples;
2. event identities have exact and empirical counterparts;
3. the same sample space can support different probability laws.

All random experiments use an explicit NumPy generator and a fixed seed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 202601
rng = np.random.default_rng(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"NumPy version: {np.__version__}")
print(f"Reproducibility seed: {SEED}")

## Example 1: running relative frequency

Under a fair-coin model, let $\Omega=\{H,T\}$, where $H$ denotes heads and $T$ denotes tails, and let $A=\{H\}$. The model assigns $\mathbb P(A)=1/2$. For $n\in\mathbb N=\{1,2,\ldots\}$, let $\omega_i\in\Omega$, $i=1,\ldots,n$, be the observed flips. Their relative frequency of $A$ is

$$
\widehat p_n(A)=\frac{1}{n}\sum_{i=1}^n \mathbf 1_A(\omega_i),
$$

where $\mathbf 1_A(\omega)$ equals one for $\omega\in A$ and zero otherwise.

Before running the cell, predict whether the running frequency will approach $1/2$ monotonically.

In [ ]:
n_flips = 5_000
heads = rng.random(n_flips) < 0.5
running_frequency = np.cumsum(heads) / np.arange(1, n_flips + 1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(np.arange(1, n_flips + 1), running_frequency, lw=1.2, label=r"$\widehat p_n(\mathrm{heads})$")
ax.axhline(0.5, color="black", ls="--", lw=1.5, label=r"model value $\mathbb{P}(\mathrm{heads})=0.5$")
ax.set(xlabel="number of flips $n$", ylabel="relative frequency", ylim=(0, 1), title="A finite sample fluctuates around the model value")
ax.legend()
plt.show()

In [ ]:
checkpoints = np.array([1, 2, 5, 10, 20, 50, 100, 500, 1_000, 5_000])
print("    n    running frequency")
for n in checkpoints:
    print(f"{n:5d}    {running_frequency[n - 1]:.4f}")

**Interpretation.** The empirical value is a relative frequency computed from this particular sequence. The horizontal line is part of the model. Agreement is approximate and finite-sample fluctuations are expected. This plot illustrates behavior under the model; it does not prove a convergence theorem. Lectures 13-14 will prove weak and bounded strong laws of large numbers under explicit assumptions on repeated sampling.

## Example 2: event algebra on a die

Let $\Omega=\{1,2,3,4,5,6\}$, $A=\{2,4,6\}$ (even), and $B=\{4,5,6\}$ (at least four). We first compute with sets, then compare exact and empirical probabilities.

In [ ]:
omega = set(range(1, 7))
A = {2, 4, 6}
B = {4, 5, 6}

print(f"Omega       = {sorted(omega)}")
print(f"A           = {sorted(A)}")
print(f"B           = {sorted(B)}")
print(f"A union B   = {sorted(A | B)}")
print(f"A intersect B = {sorted(A & B)}")
print(f"A complement  = {sorted(omega - A)}")

In [ ]:
def probability_fair_die(event):
    return len(event) / len(omega)

p_A = probability_fair_die(A)
p_B = probability_fair_die(B)
p_intersection = probability_fair_die(A & B)
p_union = probability_fair_die(A | B)

print(f"P(A union B)                         = {p_union:.4f}")
print(f"P(A) + P(B) - P(A intersect B)      = {p_A + p_B - p_intersection:.4f}")
print(f"P(A) + P(B)                         = {p_A + p_B:.4f}  (overlap counted twice)")

In [ ]:
n_rolls = 2_000
rolls = rng.integers(1, 7, size=n_rolls)
in_A = np.isin(rolls, list(A))
in_B = np.isin(rolls, list(B))

p_hat_A = in_A.mean()
p_hat_B = in_B.mean()
p_hat_intersection = (in_A & in_B).mean()
p_hat_union = (in_A | in_B).mean()

print(f"p_hat(A union B)                    = {p_hat_union:.4f}")
print(f"p_hat(A) + p_hat(B) - p_hat(A cap B) = {p_hat_A + p_hat_B - p_hat_intersection:.4f}")
print("The equality is exact for the observed sample because it is an indicator identity.")

## Example 3: one sample space, two probability laws

A fair die and a loaded die share the same sample space. They differ only in the probability law assigned to the six outcomes.

In [ ]:
faces = np.arange(1, 7)
fair_probs = np.full(6, 1 / 6)
loaded_probs = np.array([0.08, 0.10, 0.12, 0.15, 0.20, 0.35])
assert np.isclose(loaded_probs.sum(), 1.0)

n_observed = 300
observed_rolls = rng.choice(faces, size=n_observed, p=loaded_probs)
empirical_probs = np.bincount(observed_rolls, minlength=7)[1:] / n_observed

width = 0.25
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(faces - width, fair_probs, width=width, label="fair model")
ax.bar(faces, loaded_probs, width=width, label="loaded model")
ax.bar(faces + width, empirical_probs, width=width, label="observed relative frequencies")
ax.set(xlabel="die face", ylabel="probability / relative frequency", title=f"Same outcomes, different laws; {n_observed} observed rolls", xticks=faces)
ax.legend()
plt.show()

In [ ]:
event_six_observed = (observed_rolls == 6).mean()
n_simulations = 5_000
fair_simulations = rng.choice(faces, size=(n_simulations, n_observed), p=fair_probs)
fair_six_frequencies = (fair_simulations == 6).mean(axis=1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(fair_six_frequencies, bins=30, density=True, alpha=0.75, label="simulated under fair model")
ax.axvline(event_six_observed, color="crimson", lw=2, label=f"observed: {event_six_observed:.3f}")
ax.axvline(1 / 6, color="black", ls="--", lw=1.5, label="fair model value: 1/6")
ax.set(xlabel="relative frequency of rolling a six", ylabel="simulation density", title="A simple model check by simulation")
ax.legend()
plt.show()

print(f"Observed relative frequency of six: {event_six_observed:.3f}")
print(f"Central 95% of fair-model simulations: {np.quantile(fair_six_frequencies, [0.025, 0.975])}")

**Interpretation.** The two specified probability laws lead to different event probabilities and simulated frequencies. We compare their consequences, not fit or select a model from observed data. Statistical inference belongs to the separate statistics course.

## Optional extension: a stable overall frequency can hide change

The next process has success probability $0.2$ in its first half and $0.8$ in its second half. Its overall relative frequency can be near $0.5$, although the generating probabilities change over time. This known mechanism illustrates why a single aggregate frequency does not describe the whole process.

In [ ]:
n = 2_000
probabilities = np.r_[np.full(n // 2, 0.2), np.full(n // 2, 0.8)]
changing_data = rng.random(n) < probabilities
changing_running_frequency = np.cumsum(changing_data) / np.arange(1, n + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(1, n + 1), probabilities, color="crimson")
axes[0].set(xlabel="observation index", ylabel="success probability", ylim=(0, 1), title="Data-generating mechanism")
axes[1].plot(np.arange(1, n + 1), changing_running_frequency)
axes[1].axhline(0.5, color="black", ls="--")
axes[1].set(xlabel="number of observations", ylabel="running relative frequency", ylim=(0, 1), title="The aggregate can conceal change")
plt.tight_layout()
plt.show()

print(f"First-half frequency:  {changing_data[: n // 2].mean():.3f}")
print(f"Second-half frequency: {changing_data[n // 2 :].mean():.3f}")
print(f"Overall frequency:     {changing_data.mean():.3f}")

## Exercise triad and an additional proof exercise

*Presentation format:* Define the objects and assumptions, show the intermediate argument or computation, and finish with a clear conclusion. For the Python/data task, bring one informative figure and explain what it demonstrates.

1. **Conceptual:** Take $\Omega=\mathbb R$, where an outcome $\omega$ is a temperature measured in degrees Celsius. A sensor answers only the two questions $A=\{\omega\in\Omega:\omega<18\}$ and $B=\{\omega\in\Omega:\omega<22\}$. Find a sigma-algebra containing $A$ and $B$ with as few sets as possible, justify why no smaller sigma-algebra can work, and decide whether $\{20\}$ is an event.
2. **Mathematical:** Let $(\Omega,\mathcal F,\mathbb P)$ be a probability space and let $A,B\in\mathcal F$. Starting from the definition of a probability space, prove $\mathbb P(A\setminus B)=\mathbb P(A)-\mathbb P(A\cap B)$ and use it to derive the two-event union formula $\mathbb P(A\cup B)=\mathbb P(A)+\mathbb P(B)-\mathbb P(A\cap B)$.
3. **Python/data:** Use `lecture_01_examples.ipynb` as a starting point. Let $\Omega=\{1,\ldots,6\}$ and $\mathcal F=2^\Omega$. Choose nonnegative probabilities $p_1,\ldots,p_6$, not all equal, that sum to one and define $\mathbb P(\{k\})=p_k$ for $k=1,\ldots,6$. Compute the model value of $\mathbb P(A)$ for $A=\{2,4,6\}$. For two different random seeds, simulate $n=5000$ rolls and plot the relative frequency $\widehat p_m(A)$ based on the first $m$ rolls, for $m=1,\ldots,n$, together with $\mathbb P(A)$. Compare deviations at $m=50,500,5000$, explain the early and late fluctuations, and explain why the experiment illustrates but does not prove convergence.
4. **Additional mathematical exercise: De Morgan's law.** Let $\Omega$ be a sample space and let $A_k\subseteq\Omega$ for every $k\in\mathbb N$. With complements taken relative to $\Omega$, prove $\bigcap_{k=1}^{\infty}A_k=(\bigcup_{k=1}^{\infty}A_k^c)^c$. Start with an arbitrary $\omega\in\Omega$ and express membership on each side using "for every" and "there exists." Then let $\mathcal F$ be a sigma-algebra on $\Omega$ with $A_k\in\mathcal F$ for every $k$. Use the identity and the definition of a sigma-algebra to show that $\bigcap_{k=1}^{\infty}A_k\in\mathcal F$.